# 探針實驗:在 Colab 跑

流程:**設定 → 小量試跑 → 完整實驗 → 分析 → 下載結果**。日誌存在 Google Drive,Colab 斷線後重新執行同一格就會**自動續跑**。

事前準備:
1. 左側 🔑「密鑰」新增 `OPENAI_API_KEY`(或 `ANTHROPIC_API_KEY`),並開啟「筆記本存取權」。**不要把金鑰貼進程式格。**
2. 若 repo 是私有的,再新增 `GITHUB_TOKEN`(只需讀取權限)。公開 repo 不需要。

## 1. 安裝與取得程式

In [ ]:
import os, subprocess, sys
REPO = "Ting-liu0103/coop-llm"      # 改成你的 repo
tok = ""
try:
    from google.colab import userdata
    tok = userdata.get("GITHUB_TOKEN") or ""
except Exception:
    pass
url = f"https://{tok + '@' if tok else ''}github.com/{REPO}.git"
if not os.path.isdir("/content/coop-llm"):
    subprocess.run(["git", "clone", "--depth", "1", url, "/content/coop-llm"], check=True)
else:
    subprocess.run(["git", "-C", "/content/coop-llm", "pull", "--ff-only"], check=True)
os.chdir("/content/coop-llm")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[openai,anthropic,plots,dev]"], check=True)
print("OK")

## 2. 自我檢查(78 項測試,不花錢)

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "-q"], capture_output=True, text=True)
print(r.stdout[-400:])
assert r.returncode == 0, "測試沒全過,先不要往下跑"

## 3. 設定模型與金鑰
填下面的變數。`provider` 可選 `openai_compat`(OpenAI 與相容服務)或 `anthropic`。
**型號請自己填**(這裡的字串是占位)。要比較兩個模型就保留兩項;只想先測一個就刪掉另一項。

In [ ]:
import yaml
from google.colab import userdata

MODELS = [
    {"name": "api_small", "provider": "openai_compat", "model": "REPLACE_WITH_MODEL_ID",
     "api_key_env": "OPENAI_API_KEY"},
    # 第二個模型(例如另一家的 API 或開源模型)。不用就整段刪掉。
    # {"name": "model_b", "provider": "anthropic", "model": "REPLACE_WITH_MODEL_ID",
    #  "api_key_env": "ANTHROPIC_API_KEY"},
]
for m in MODELS:
    assert "REPLACE" not in m["model"], f"請先填 {m['name']} 的型號"
    os.environ[m["api_key_env"]] = userdata.get(m["api_key_env"])   # 從 Colab 密鑰讀取
yaml.safe_dump({"models": MODELS}, open("configs/models_run.yaml", "w"), allow_unicode=True)
print([m["name"] for m in MODELS])

## 4. 日誌存到 Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
LOGS = "/content/drive/MyDrive/coop-llm-logs/probe"
os.makedirs(LOGS, exist_ok=True)
import glob
CONFIGS = sorted(glob.glob("configs/probe/*.yaml"))
print(CONFIGS)

## 5. 小量試跑(每格 2 場)
先確認格式正常、算出費用。跑完看下一格的解析失敗率與 token 用量。

In [ ]:
def run_all(reps, out, start_rep=0):
    for c in CONFIGS:
        r = subprocess.run([sys.executable, "-m", "coop_llm.runner", "--condition", c, "--models",
                            "configs/models_run.yaml", "--reps", str(reps), "--start-rep", str(start_rep),
                            "--out", out, "--concurrency", "8"], capture_output=True, text=True)
        print(c.split("/")[-1], "→", r.stdout.strip().splitlines()[-1] if r.stdout.strip() else r.stderr[-300:])

PILOT = "/content/drive/MyDrive/coop-llm-logs/pilot"
run_all(2, PILOT)

In [ ]:
import pandas as pd
from coop_llm.analysis import load_logs, parse_failure_report
rounds, chats, summary = load_logs(PILOT)
print("完成場次:", summary["game_id"].nunique())
print(parse_failure_report(rounds).to_string())
print("輸入 token 合計:", int(rounds["tokens_in"].sum()), " 輸出 token 合計:", int(rounds["tokens_out"].sum()))
print("看一段想法:", rounds["thought"].dropna().iloc[0][:200])

**檢查清單再往下:**
- 解析失敗率(上表 `fail_rate`)明顯 > 5% → 先回來調提示詞,不要放大。
- 用 token 合計 × (完整場數 ÷ 試跑場數) × 模型單價,估完整實驗費用。
- 想法內容看起來合理(沒有空白、沒有亂碼)。

## 6. 完整實驗(每格 20 場,可重複執行續跑)

In [ ]:
run_all(20, LOGS)

## 7. 分析

In [ ]:
out = subprocess.run([sys.executable, "-m", "coop_llm.analyze_probe", "--logs", LOGS,
                      "--out", "/content/drive/MyDrive/coop-llm-logs/results"], capture_output=True, text=True)
print(out.stdout[-600:], out.stderr[-600:])

In [ ]:
from IPython.display import Image, display
import glob
for p in sorted(glob.glob("/content/drive/MyDrive/coop-llm-logs/results/quality_map_*.png")):
    display(Image(p))

結果檔都在 Drive 的 `coop-llm-logs/results/`(`di_pairs.csv`、`comparisons_*.csv`、圖)。